### Proceso para comprobar links de redireccion al geoportal para evitar errores al eliminar capas

In [15]:
import os
import re
import time
import pandas as pd
import requests
import urllib3

urllib3.disable_warnings(urllib3.exceptions.InsecureRequestWarning)
os.environ["CURL_CA_BUNDLE"] = ""
os.environ["REQUESTS_CA_BUNDLE"] = ""
os.environ["SSL_CERT_FILE"] = ""

BASE_URL = "https://tiles.tierradelfuego.gob.ar"
TIMEOUT_SEGUNDOS = 8
DELAY_ENTRE_REQUESTS = 0.15
MAX_REINTENTOS = 2  # Reintenta si hay timeout para evitar falsos positivos


def test_dataset_real(url):
    url = str(url).strip()
    if not url or url.lower() in ["nan", "none", ""]:
        return None, "VACIO"

    session = requests.Session()
    session.verify = False

    # 1. Caso URLs con ID de catálogo: validar directo contra la API REST
    match_id = re.search(r"/dataset/(\d+)", url)
    if match_id:
        dataset_id = match_id.group(1)
        api_url = f"{BASE_URL}/api/v2/datasets/{dataset_id}/"

        for intento in range(MAX_REINTENTOS + 1):
            try:
                r = session.get(api_url, timeout=TIMEOUT_SEGUNDOS)
                if r.status_code == 200:
                    if "application/json" in r.headers.get("Content-Type", ""):
                        return True, "HTTP 200 (API OK)"
                    return False, "Error (No es JSON)"
                elif r.status_code == 404:
                    return False, "HTTP 404 (No existe)"
                else:
                    return False, f"HTTP {r.status_code}"
            except (requests.exceptions.Timeout, requests.exceptions.ConnectionError):
                if intento < MAX_REINTENTOS:
                    time.sleep(0.5)
                    continue
                return False, "Timeout persistente"
            except Exception as e:
                return False, f"Error: {type(e).__name__}"

    # 2. Caso URLs directas (WFS, CSW, datasets directos)
    for intento in range(MAX_REINTENTOS + 1):
        try:
            with session.get(
                url, timeout=TIMEOUT_SEGUNDOS, allow_redirects=True, stream=True
            ) as r:
                if r.status_code != 200:
                    return False, f"HTTP {r.status_code}"

                fragmento = (
                    r.raw.read(1024).decode("utf-8", errors="ignore").lower()
                )
                patrones_error = [
                    "página no encontrada",
                    "pagina no encontrada",
                    "not found",
                    "no se encontró el recurso",
                    "error 404",
                    "error 500",
                ]

                if any(p in fragmento for p in patrones_error):
                    return False, "Error 404 Camuflado (HTML Error)"

                return True, "HTTP 200"

        except (requests.exceptions.Timeout, requests.exceptions.ConnectionError):
            if intento < MAX_REINTENTOS:
                time.sleep(0.5)
                continue
            return False, "Timeout persistente"
        except Exception as e:
            return False, f"Error: {type(e).__name__}"

In [16]:
print(f"Cargando hoja '{NOMBRE_HOJA}' desde '{ARCHIVO_EXCEL}'...")
df_raw = pd.read_excel(ARCHIVO_EXCEL, sheet_name=NOMBRE_HOJA, dtype=str, engine="openpyxl")
df_raw.columns = df_raw.columns.str.strip()

# 1. Detectar columna de links dinámicamente
col_link = next(
    (c for c in df_raw.columns if any(k in c.lower() for k in ["link", "url", "enlace", "catalogue", "dataset"])), 
    None
)

if not col_link:
    print("Columnas encontradas:", df_raw.columns.tolist())
    raise KeyError("No se encontró la columna de links. Revisa los nombres listados arriba.")

# 2. Detectar columna identificadora (nombre/objeto/capa/id)
col_id = next(
    (c for c in df_raw.columns if any(k in c.lower() for k in ["objeto", "nombre", "capa", "titulo", "id"])), 
    df_raw.columns[0]
)

# 3. Filtrar solo filas con datos
df_filtrado = df_raw[
    df_raw[col_link].notna() & 
    (df_raw[col_link].str.strip() != "") & 
    (df_raw[col_link].str.lower() != "nan")
].copy()

print(f"Columna de links detectada: '{col_link}'")
print(f"Columna identificadora: '{col_id}'")
print(f"Total registros en la hoja: {len(df_raw)}")
print(f"Total filas con link para validar: {len(df_filtrado)}")
df_filtrado[[col_id, col_link]].head(5)

Cargando hoja 'Objetos' desde 'Catalogo_Normalizado.xlsx'...
Columna de links detectada: 'Link'
Columna identificadora: 'ID_Subclase_FK'
Total registros en la hoja: 298
Total filas con link para validar: 87


,ID_Subclase_FK,Link
2,0101,https://tiles.tierradelfuego.gob.ar/catalogue/...
10,0102,https://tiles.tierradelfuego.gob.ar/catalogue/...
11,0102,https://tiles.tierradelfuego.gob.ar/catalogue/...
21,0104,https://tiles.tierradelfuego.gob.ar/catalogue/...
25,0105,https://tiles.tierradelfuego.gob.ar/catalogue/...


In [17]:
total = len(df_filtrado)
print(f"Iniciando comprobación segura de {total} enlaces...\n")

estados_ok = []
respuestas_http = []

for idx, (_, fila) in enumerate(df_filtrado.iterrows(), start=1):
    url = fila[col_link]
    ok, estado = test_dataset_view(url)
    
    estados_ok.append(ok)
    respuestas_http.append(estado)
    
    # Imprimir inmediatamente si da error o no responde 200
    if not ok:
        print(f"[{idx}/{total}] ERROR -> [{estado}] {fila[col_id]}")
        print(f"    URL: {url}\n")
    elif idx % 20 == 0 or idx == total:
        print(f"Progreso: {idx}/{total} enlaces procesados...")
        
    time.sleep(DELAY_ENTRE_REQUESTS)

# Agregar resultados al DataFrame
df_filtrado["Activo"] = estados_ok
df_filtrado["Estado_HTTP"] = respuestas_http

# Filtrar enlaces con error
df_errores = df_filtrado[df_filtrado["Activo"] == False].copy()

print("\n" + "=" * 80)
print(f" RESUMEN: {len(df_errores)} LINKS ROTOS O INACCESIBLES DE {total}")
print("=" * 80)

if df_errores.empty:
    print("Todos los enlaces verificados devolvieron HTTP 200 correctamente.")
else:
    cols_salida = [col_id, col_link, "Estado_HTTP"]
    df_errores[cols_salida].to_csv(OUTPUT_ERRORES_CSV, index=False, encoding="utf-8-sig")
    print(f"Archivo con errores generado: {OUTPUT_ERRORES_CSV}")
    
    # Mostrar tabla en Jupyter
    display(df_errores[cols_salida])

Iniciando comprobación segura de 87 enlaces...

Progreso: 20/87 enlaces procesados...
Progreso: 40/87 enlaces procesados...
Progreso: 60/87 enlaces procesados...
Progreso: 80/87 enlaces procesados...
Progreso: 87/87 enlaces procesados...

 RESUMEN: 0 LINKS ROTOS O INACCESIBLES DE 87
Todos los enlaces verificados devolvieron HTTP 200 correctamente.
